# Speech translation with OWSM-CTC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/st_demo.ipynb)

English speech in, text in another language out — the same checkpoint that
transcribes. You choose the target by asking for a different task.

CPU is enough. The checkpoint is 4 GB and cached after the first run.


## Install


In [ ]:
%pip install -q "espnet==202610.post1" espnet_model_zoo librosa


## A recording to translate


In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("sample.wav", sr=16000)
display(Audio(speech, rate=rate))


## One model, four languages

The task symbol carries the target: `<asr>` transcribes, `<st_deu>`
translates to German, and so on. The model's token list is where the
available pairs are written, so ask it rather than a table.


In [ ]:
from espnet2.bin.s2t_inference import Speech2Text

s2t = Speech2Text.from_pretrained("espnet/owsm_ctc_v4_1B", device="cpu")

targets = [t for t in s2t.s2t_model.token_list if t.startswith("<st_")]
print(f"{len(targets)} translation targets, e.g. {targets[:6]}")


In [ ]:
for task in ("<asr>", "<st_deu>", "<st_fra>", "<st_zho>"):
    segments = s2t.decode_long(
        "sample.wav", lang_sym="<eng>", task_sym=task
    )
    print(f"{task:10} {' '.join(text for _, _, text in segments)}")


## Where next

- **From the terminal**: `espnet translate sample.wav --to deu`
- **In the browser**: the [owsm-ctc-v4 Space](https://huggingface.co/spaces/espnet/owsm-ctc-v4) has the same menu
- **Transcription** with the same model: [`asr_demo.ipynb`](asr_demo.ipynb)
- **Simultaneous translation**, which is a different model and a longer
  story: [`../Courses/CMUSpeechTechnology26S/speech_translation.ipynb`](../Courses/CMUSpeechTechnology26S/speech_translation.ipynb)
